# Практическое занятие №4: Оптимизация мультиклассовой модели

**Задача:** Улучшить мультиклассовую модель из Практики 3 с помощью SMOTE, кривых обучения и GridSearchCV.  
**Датасет:** Wine (scikit-learn)  
**Методы:** SMOTE, Learning Curve, Pipeline + GridSearchCV

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn.model_selection import train_test_split, learning_curve, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import warnings; warnings.filterwarnings('ignore')

## Загрузка и подготовка данных

In [ ]:
wine = datasets.load_wine()
X = wine.data
y = wine.target
target_names = wine.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 1. Применение продвинутых методов (SMOTE)

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Распределение классов до SMOTE (Train): {np.bincount(y_train)}")
print(f"Распределение классов после SMOTE (Train): {np.bincount(y_train_smote)}")

rf_smote = RandomForestClassifier(random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = rf_smote.predict(X_test)

macro_f1_smote = f1_score(y_test, y_pred_smote, average='macro')
print("\nОтчет классификации (RandomForest + SMOTE):")
print(classification_report(y_test, y_pred_smote, target_names=target_names))

## 2. Диагностика с помощью кривых обучения

In [ ]:
print("Построение кривых обучения. Пожалуйста, подождите...")
train_sizes, train_scores, test_scores = learning_curve(
    RandomForestClassifier(random_state=42), X_train_smote, y_train_smote,
    cv=5, scoring='f1_macro', n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5)
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, color='steelblue', marker='o', markersize=6, label='Обучающая выборка')
plt.fill_between(train_sizes, train_mean + train_std, train_mean - train_std, alpha=0.15, color='steelblue')
plt.plot(train_sizes, test_mean, color='darkorange', linestyle='--', marker='s', markersize=6,
         label='Валидационная выборка')
plt.fill_between(train_sizes, test_mean + test_std, test_mean - test_std, alpha=0.15, color='darkorange')
plt.title('Кривые обучения (Learning Curve): RandomForest + SMOTE')
plt.xlabel('Объем обучающей выборки')
plt.ylabel('Macro-F1 Score')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('learning_curve.png')
plt.show()

print("Анализ: Модель показывает высокий скор на трейне (~1.0), валидационная кривая растет. Явного жесткого переобучения нет, но сбор дополнительных данных мог бы еще больше сблизить кривые.")

## 3. Комплексная настройка (Pipeline + GridSearchCV)

In [ ]:
pipeline = ImbPipeline(steps=[
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

param_grid = {
    'rf__max_depth': [5, 10, None],
    'rf__n_estimators': [50, 100]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1)

print("Запуск GridSearchCV...")
grid_search.fit(X_train, y_train)

print("\nЛучшая комбинация гиперпараметров:")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")

best_model = grid_search.best_estimator_
y_pred_grid = best_model.predict(X_test)
macro_f1_grid = f1_score(y_test, y_pred_grid, average='macro')

## Итоговая таблица результатов

In [ ]:
base_f1 = 0.9815

data = [
    ['Базовая модель (практика 3)', f"{base_f1:.4f}"],
    ['+ SMOTE (Без тюнинга)', f"{macro_f1_smote:.4f}"],
    ['+ Pipeline + GridSearchCV', f"{macro_f1_grid:.4f}"]
]
results_df = pd.DataFrame(data, columns=['Шаг', 'Макро-F1 (test)'])
print(results_df.to_string(index=False))